# Pretrained models

## Setup

In [ ]:
import sys
from pathlib import Path

BRANCH = "master"

if "google.colab" in sys.modules:
    REPO = Path("/content/CNN_for_multilabel_classification")
    if REPO.exists():
        !git -C {REPO} pull -q
    else:
        !git clone -q -b {BRANCH} https://github.com/vumenira/CNN_for_multilabel_classification.git {REPO}
    %pip install -q -r {REPO}/requirements-colab.txt
else:
    REPO = Path.cwd().parent  # running locally from notebooks/

sys.path.insert(0, str(REPO))

from src import colab
colab.setup()

import pandas as pd
import torch
from torchmetrics.functional.classification import multilabel_f1_score

from src.classifier import predict
from src.pretrained import train
from src.config import DATA_DIR, FOLDS_PATH, LABELS, ROOT
from src.postprocess import add_parents, best_thresholds, blend_duplicates, hflip, phash

folds = pd.read_csv(FOLDS_PATH).set_index("image_id")


def val_f1(pred: pd.DataFrame, y: pd.DataFrame) -> float:
    return multilabel_f1_score(torch.tensor(pred.to_numpy(dtype="float32")), torch.tensor(y.to_numpy()), len(LABELS)).item()

Mounted at /content/drive
DATA_DIR=/content/data
MODELS_DIR=/content/models
RESULTS_DIR=/content/drive/MyDrive/dl_lab2/results


## Experiment 1: ConvNeXt-B in22k, 384

In [3]:
import time

from src import tracking
from src.config import RESULTS_DIR
from src.sync import SyncCheckpoints

SMOKE = False

RUN_NAME = "smoke" if SMOKE else "convnext_b_384_fold0_head_posw"
SMOKE_KWARGS = dict(max_epochs=2, trainer_kwargs={"limit_train_batches": 30, "limit_val_batches": 10}) if SMOKE else {}
SAVE_DIR = RESULTS_DIR / RUN_NAME
last = SAVE_DIR / "last.ckpt"
resume = last.exists()
if resume:
    colab.restore_sqlite(tracking.MLRUNS_DIR / "mlflow.db", SAVE_DIR / "mlflow.db")
print(f"{'resuming from' if resume else 'new run, saving to'} {SAVE_DIR}")

start = time.time()
trainer = train(
    "convnext_base.fb_in22k_ft_in1k_384",
    size=384,
    cache=False,
    callbacks=[SyncCheckpoints(SAVE_DIR)],
    ckpt_path=last if resume else None,
    run_id=(SAVE_DIR / "run_id.txt").read_text() if resume else None,
    **SMOKE_KWARGS,
)
print(f"{time.time() - start:.0f} s")
if torch.cuda.is_available():
    print(f"max GPU memory {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")
dm = trainer.datamodule
RUN_ID = trainer.logger.run_id
best = SAVE_DIR / "best.ckpt"

INFO: Seed set to 42
INFO:lightning.fabric.utilities.seed:Seed set to 42


new run, saving to /content/drive/MyDrive/dl_lab2/results/convnext_b_384_fold0


INFO: Using 16bit Automatic Mixed Precision (AMP)
INFO:lightning.pytorch.utilities.rank_zero:Using 16bit Automatic Mixed Precision (AMP)
INFO: GPU available: True (cuda), used: True
INFO:lightning.pytorch.utilities.rank_zero:GPU available: True (cuda), used: True
INFO: TPU available: False, using: 0 TPU cores
INFO:lightning.pytorch.utilities.rank_zero:TPU available: False, using: 0 TPU cores
INFO: 💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO:lightning.pytorch.utilities.rank_zero:💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
INFO: LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
INFO:lightning.pytorch.accelerators.cuda:LOCAL_R

┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type                       ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ ConvNeXt                   │ 87.6 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss          │      0 │ train │     0 │
│ 2 │ train_f1  │ MultilabelF1Score          │      0 │ train │     0 │
│ 3 │ val_f1    │ MultilabelF1Score          │      0 │ train │     0 │
│ 4 │ val_ap    │ MultilabelAveragePrecision │      0 │ train │     0 │
└───┴───────────┴────────────────────────────┴────────┴───────┴───────┘

Trainable params: 87.6 M                                                                                           
Non-trainable params: 0                                                                                            
Total params: 87.6 M                                                                                               
Total estimated model params size (MB): 350.286                                                                    
Modules in train mode: 469                                                                                         
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

/usr/local/lib/python3.13/dist-packages/lightning/pytorch/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


... (2 outputs left out)

INFO: 
Detected KeyboardInterrupt, attempting graceful shutdown ...
INFO:lightning.pytorch.utilities.rank_zero:
Detected KeyboardInterrupt, attempting graceful shutdown ...


ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/call.py", line 49, in _call_and_handle_interrupt
    return trainer_fn(*args, **kwargs)
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/trainer.py", line 630, in _fit_impl
    self._run(model, ckpt_path=ckpt_path, weights_only=weights_only)
    ~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/trainer.py", line 1079, in _run
    results = self._run_stage()
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/trainer/trainer.py", line 1123, in _run_stage
    self.fit_loop.run()
    ~~~~~~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py", line 217, in run
    self.advance()
    ~~~~~~~~~~~~^^
  File "/usr/local/lib/python3.13/dist-packages/lightning/pytorch/loops/fit_loop.py", line 469, in advance
    self.

TypeError: object of type 'NoneType' has no len()

In [ ]:
val = predict(trainer, dataloaders=dm.val_dataloader(), ckpt_path=best)
test = predict(trainer, datamodule=dm, ckpt_path=best)
val.to_csv(SAVE_DIR / "val_probs.csv")
test.to_csv(SAVE_DIR / "test_probs.csv")
y = folds.loc[val.index, LABELS]
val_f1(val, y)

In [ ]:
path = SAVE_DIR / "submission.csv"
(test >= 0.5).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
if not SMOKE:
    # !kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384"
    pass

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4

### TTA hflip

In [ ]:
val_tta = (val + predict(trainer, dataloaders=dm.loader(hflip(dm.val_ds)), ckpt_path=best)) / 2
test_tta = (test + predict(trainer, dataloaders=dm.loader(hflip(dm.test_ds)), ckpt_path=best)) / 2
val_f1(val_tta, y)

### pHash ≤ 2

In [ ]:
is_val = (folds.fold == dm.fold).to_numpy()
hashes = phash(DATA_DIR / "train" / f"{i}.jpg" for i in folds.index)
labels = folds[LABELS].to_numpy()

val_dup = blend_duplicates(val_tta, hashes[folds.index.get_indexer(val.index)], hashes[~is_val], labels[~is_val])
test_dup = blend_duplicates(test_tta, phash(DATA_DIR / "test" / f"{i}.jpg" for i in test.index), hashes, labels)
val_f1(val_dup, y)

### Per-class thresholds

In [ ]:
thresholds = best_thresholds(val_dup, y)
print(thresholds.round(2).to_dict())
val_f1(val_dup >= thresholds, y)

### Hierarchy

In [ ]:
val_f1(add_parents(val_dup >= thresholds), y)

### Submit

In [ ]:
path = SAVE_DIR / "submission_post.csv"
add_parents(test_dup >= thresholds).astype(int).rename_axis("image_id").reset_index().to_csv(path, index=False)
if not SMOKE:
    # !kaggle competitions submit -c cats-and-dogs -f {path} -m "{RUN_ID} convnext_base in22k 384 + post"
    pass

In [ ]:
!kaggle competitions submissions -c cats-and-dogs | head -4

## Finish: wait until Drive has every file

In [ ]:
if colab.in_colab():
    from google.colab import drive
    drive.flush_and_unmount()